# Chess Bot — Evaluation on Colab (GPU)

Runs the **eval pipeline** (`eval/scrape_eval_data.py` → `eval/prepare_eval.py` → `eval/eval_new.py`) on a T4 GPU.

Prerequisites on Drive (from training run):
- `src/` (board_utils.py, dataset.py, model.py)
- `data/processed/positions.npy` (needed by `prepare_eval.py` to dedup train/eval)
- `checkpoints/*.pt` (the models to evaluate)
- `eval/` (eval_new.py, prepare_eval.py, scrape_eval_data.py)
- `requirements.txt`

The eval PGN (`lichess_elite_2024-06`) is downloaded fresh into `eval/data/` by `scrape_eval_data.py`.

In [ ]:
# Cell 1: Mount Drive and cd to the project root on Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/Chess_bot')
print("Working directory:", os.getcwd())
print("Files:", os.listdir('.'))

In [ ]:
# Cell 2: Install dependencies (torch + numpy already in Colab)
!pip install python-chess==1.999 tqdm requests -q --upgrade
print("Dependencies installed!")

In [ ]:
# Cell 3: Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
# Expected: CUDA available: True, GPU: Tesla T4, VRAM: 15.8 GB

In [ ]:
# Cell 4: Download the held-out eval PGN (lichess_elite_2024-06) into eval/data/
# scrape_eval_data.py writes to eval/data/ and skips if the PGN already exists.
import subprocess, os
if not os.path.exists('eval/data/lichess_elite_2024-06.pgn'):
    result = subprocess.run(['python', 'eval/scrape_eval_data.py'], capture_output=False)
else:
    print("Eval PGN already exists: eval/data/lichess_elite_2024-06.pgn")

In [ ]:
# Cell 5: Build the held-out eval dataset (parse + dedup against training positions).
# Reads:  data/processed/positions.npy  (training set, for tensor-hash dedup)
#         eval/data/lichess_elite_2024-06.pgn
# Writes: eval/data/processed_eval/{positions,moves,values,mover_elo}.bin + meta.json
#
# This is the slow cell (~10-20 min). It only needs to run once; the .bin files
# persist on Drive and later runs skip straight to Cell 6.
import subprocess
if not os.path.exists('eval/data/processed_eval/meta.json'):
    result = subprocess.run(['python', 'eval/prepare_eval.py'], capture_output=False)
else:
    print("Eval dataset already built: eval/data/processed_eval/meta.json")

In [ ]:
# Cell 6: Run evaluation (top-1/top-5 by ELO bin + estimated playing strength).
# Loads every checkpoint in checkpoints/best_model*.pt and compares them.
# ~1-2 min on T4 for the capped 200k/bin eval set.
import subprocess
result = subprocess.run(['python', 'eval/eval_new.py'], capture_output=False)